# Human-in-the-Loop : Portes pré-action, classification des risques et journal d’audit

Le README de cette leçon introduit le Human-in-the-Loop avec un court extrait qui demande à l’utilisateur `APPROVE` ou `REJECT` après que l’agent ait déjà produit une réponse. Ce modèle est un bon point de départ, mais les implémentations HITL en production nécessitent couramment trois éléments supplémentaires :

1. Une **porte pré-action** qui s’exécute **avant** que l’agent réalise une étape risquée, afin que le coût, l’irréversibilité et la latence restent sous contrôle.
2. Une **classification des risques**, pour que les actions à faible risque s’exécutent automatiquement, les actions à risque moyen soient approuvées par lots, et seules les actions à haut risque soient bloquées par une intervention humaine.
3. Un **journal d’audit plus une boucle de révision**, de sorte que chaque décision de porte soit enregistrée au format JSONL, et qu’un rejet relance l’agent avec une raison structurée au lieu d’afficher simplement `Revising...`.

Ce notebook construit chacun de ces éléments au-dessus des mêmes primitives que `06-system-message-framework.ipynb`. Il s’exécute de bout en bout en `DEMO_MODE = True` (aucune saisie interactive requise) ou avec de réelles invites `input()` lorsque `DEMO_MODE = False`. Note : en DEMO_MODE, la nouvelle tentative du troisième objectif est scriptée pour que la mécanique de la boucle soit visible de bout en bout. Une re-classification effective impulsée par révision nécessite `DEMO_MODE = False` et un opérateur.

**Hors périmètre (traité dans d’autres leçons) :** authentification et contrôle d’accès (menace 2 du README de la Leçon 06), middleware d’appel d’outil (analyse approfondie MAF Leçon 14), modèles de débat multi-agent.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Modèle 1 : Portail pré-action

L'extrait HITL du README appelle d'abord l'agent, puis demande à l'utilisateur d'approuver le résultat. Il s'agit d'un flux **post-action**. L'agent a déjà été exécuté, donc le coût d'appel du LLM est déjà payé, et tout effet secondaire (email envoyé, ligne de base de données écrite, commentaire posté) s'est déjà produit.

Un flux **pré-action** insère le portail avant que l'agent n'effectue l'étape risquée. L'agent propose l'action, le portail décide d'exécuter ou non, et ce n'est qu'après approbation que l'effet secondaire se produit.

| Aspect | Approbation post-action (extrait README) | Portail pré-action (ce notebook) |
|---|---|---|
| Quand l'approbation s'exécute-t-elle ? | Après que l'agent a produit le résultat | Avant toute exécution d'effet secondaire |
| Coût LLM en cas de rejet | Déjà payé | Payé seulement pour la proposition, pas pour l'action |
| Effets secondaires irréversibles | Possibles (l'action a déjà eu lieu) | Prévenus |
| Clarté de l'audit | L'approbation est une instruction d'affichage | L'approbation est un enregistrement JSON avec horodatage, action, raison |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Modèle 2 : Classification des risques

Toutes les actions ne nécessitent pas une approbation humaine. Une consultation en lecture seule via une API publique a des enjeux différents que l’envoi d’un courriel à un client. Traiter les deux de la même manière gaspille l’attention de l’opérateur et ralentit l’agent.

Un modèle simple à 3 niveaux :

| Niveau | Exemples | Processus d’approbation |
|---|---|---|
| `faible` (lecture seule) | Rechercher dans une base de connaissances, consulter des options de vol, récupérer une page web publique | Exécution automatique, journalisée pour audit |
| `moyen` (mutation peu coûteuse) | Mettre en cache un résultat, incrémenter un compteur, programmer un rappel | Exécution automatique, mais revue groupée quotidienne |
| `élevé` (externe ou irréversible) | Envoyer un courriel, débiter une carte, publier sur un canal public | Blocage en attente d’approbation humaine |

Voici une classification. Les systèmes en production utilisent souvent des niveaux plus granulaires (par exemple, les niveaux d’autorisation AWS IAM, les niveaux d’accès basés sur les rôles). La version à 3 niveaux ci-dessous est la plus petite version utile pour un agent qui mélange des actions en lecture seule et des actions avec effets secondaires.

Le classificateur ci-dessous utilise des heuristiques basées sur des mots-clés pour que la démo reste déterministe et peu coûteuse. Dans un système en production, vous remplaceriez cela par un classificateur appris ou un moteur de règles.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Modèle 3 : Journal d'audit et boucle de révision

Un `print("Response approved.")` n'est pas un journal d'audit. Pour la confiance, chaque décision de porte doit être enregistrée en tant qu'événement structuré que vous pouvez ensuite interroger, rejouer ou joindre à une revue d’incident.

Deux éléments :

1. **JSONL en ajout seul.** Une ligne par décision, avec horodatage, action, niveau, décision, raison. Facile à extraire avec grep, facile à envoyer vers un vrai magasin de journaux plus tard.  
2. **Boucle de révision en cas de rejet.** Lorsque la porte retourne `deny`, l'agent se relance en se fournissant la raison du rejet dans le contexte, afin que la proposition suivante puisse éviter le problème.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Ressources supplémentaires

Plusieurs autres projets publics implémentent des variantes de ces modèles HITL. Comparez les approches pour trouver celle qui convient le mieux à votre stack :

- **LangChain** wrappers d'outils human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)) : wrappers d'outils prêts à l'emploi qui suspendent l'exécution pour une entrée humaine.
- **AutoGen** `UserProxyAgent` ([docs v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop) ; AutoGen v0.4+ a restructuré cela) : utilise un rôle d'agent spécifiquement pour représenter l'humain dans des conversations multi-agents.
- **Semantic Kernel** filtres de fonction ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)) : middleware qui s'exécute autour de chaque appel de fonction, adapté à la logique de contrôle.

Chaque projet gère différemment les trois sous-modèles : LangChain les encapsule en tant qu'outils, AutoGen utilise un rôle d'agent, Semantic Kernel utilise des filtres middleware. Lisez une ou deux implémentations dans leur intégralité avant de choisir une conception pour votre propre agent.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
